In [28]:
import sys
import os
import pandas as pd
import numpy as np
import warnings
import scipy.stats as st
import glob
import re
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

warnings.filterwarnings('ignore')

In [31]:
experiment_csv = os.path.join('experiment_tables', "experiment_table_20251111_113627.pkl")
df_experiments = pd.read_pickle(experiment_csv)  # Use read_csv if the file is in CSV format
folder='outputs' 
feature_files = glob.glob(os.path.join(folder, "*.json"))
print(feature_files)
df_pre_patch, df_pre_extracted, df_pre_contrastive, df_pre_handcrafted = file_IO.preprocess_experiment_logs(source_path)
df_visualization = file_IO.get_reference_table_for_experiments(df_pre_patch, df_pre_extracted, im_show=False,im_plot=False)

['outputs\\results_standard_patches.json', 'outputs\\results_standard_patches_no_scaling.json', 'outputs\\test_standard_patches.json', 'outputs\\test_standard_patches_no_scaling.json']


In [32]:
all_ensembling_approaches = ['individual','ensembled','ensembled_weighted','ensembled_most_probable','ensembled_writers']
best_metric='ensembled_weighted'
all_groups=['english,different',
 'english,same',
 'arabic,different',
 'arabic,same',
 'english,all',
 'arabic,all',
 'all,different',
 'all,same']
IF_OOF = ['IF','OOF']
all_metrics = ['Mean','Min','Max','Variance','Median','Confidence Interval','Generalization Gap']
experiment_results={}

# standard patches

In [33]:
experiment='outputs\\results_standard_patches.json'

In [36]:
df_standard_patches=process_results(experiment, df_pre_patch,box_and_whiskers=False)
experiment_results['standard_patches'] = df_standard_patches

In [45]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(df_standard_patches.sort_values(by='accuracies', ascending=False))
save_table_as_image(df_standard_patches,significant_digits=4)

,FE model,accuracies,accuracies_std,ind_accuracies,ind_accuracies_std
63,BEiT-Large,0.767826,0.061211,0.718713,0.047689
22,BEiT-Base,0.765025,0.055745,0.711188,0.036050
25,clip-vit-large-patch14-un,0.741010,0.044030,0.685979,0.045204
46,convnext_large,0.734883,0.061863,0.698615,0.041903
23,clip-vit-base-patch16,0.730388,0.045668,0.677531,0.045884
19,DeiT-Small,0.730357,0.051938,0.673528,0.039391
18,swin_s,0.729526,0.062252,0.687377,0.044621
45,efficientnet_v2_s,0.727894,0.061601,0.689680,0.052321
13,trocr-large-stage1,0.724969,0.064788,0.689101,0.056321
47,convnext_base,0.724415,0.076435,0.690203,0.053959


Saved table image to: df_standard_patches_table_sig4.png


# standard patches un

# easy access

In [40]:
def reload_modules():
    import importlib
    import utils.file_IO as file_IO
    import utils.experiment_analysis_utils as experiment_analysis_utils
    
    importlib.reload(file_IO)
    importlib.reload(experiment_analysis_utils)

    return file_IO, experiment_analysis_utils
file_IO, experiment_analysis_utils = reload_modules()

In [44]:
from pandas.api.types import is_numeric_dtype

def process_results(experiment, df_pre_patch,box_and_whiskers=True):
    df = file_IO.assemble_csv_from_log(experiment)
    df=experiment_analysis_utils.extends_with_info_from_source(df,df_pre_patch,columns=['unique name', 'type'])
    features_of_interest = ['FE model','type']
    df = df[features_of_interest+['cross_val_accuracies','cross_val_subgroup_accuracies']]
    #df_exp=expand_accuracies(df, all_ensembling_approaches,all_groups)
    df_exp = experiment_analysis_utils.get_crossval_results(df)
    #selected_accuracies = select_accuracies(all_ensembling_approaches, ['OOF','IF'], ['Mean','Generalization Gap'], ['all,all','english,all','arabic,all'])
    selected_accuracies = ['individual_accuracies', 'ensembled_weighted_accuracies']
    #print(selected_accuracies)
    #ensembling_types=['ensembled'],IF_OOF=['OOF'],metrics=['Mean'],groups=['all,all','english,all']
    df=df_exp[features_of_interest+selected_accuracies]
    accuracies=df_exp['ensembled_weighted_accuracies'].apply(lambda x: np.mean(x))
    accuracies_std=df_exp['ensembled_weighted_accuracies'].apply(lambda x: np.std(x))
    ind_accuracies=df_exp['individual_accuracies'].apply(lambda x: np.mean(x))
    ind_accuracies_std=df_exp['individual_accuracies'].apply(lambda x: np.std(x))
    df_processed = pd.DataFrame({
        'FE model': df_exp['FE model'],
        'accuracies': accuracies,
        'accuracies_std': accuracies_std,
        'ind_accuracies': ind_accuracies,
        'ind_accuracies_std': ind_accuracies_std
    })
    if box_and_whiskers:
        experiment_analysis_utils.box_and_whiskers(df, type='squares', accuracy_col='individual_accuracies',
                                                   title='Distribution of Accuracies per FE Model, standard body dataset')
    return df_processed
def save_table_as_image(df_standard_patches, significant_digits=4):
    import matplotlib.pyplot as plt

    _sorted = df_standard_patches.sort_values(by='accuracies', ascending=False).reset_index(drop=True)

    def format_sig(x, sig):
        if pd.isna(x):
            return ""
        return f"{x:.{sig}g}"

    formatted = _sorted.copy()
    for col in formatted.columns:
        if is_numeric_dtype(formatted[col]):
            formatted[col] = formatted[col].apply(lambda v: format_sig(v, significant_digits))

    rows, cols = formatted.shape
    fig_w = max(10, cols * 2.2)
    fig_h = min(40, 0.35 * rows + 1.5)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    table = ax.table(
        cellText=formatted.values,
        colLabels=formatted.columns,
        loc='center',
        cellLoc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.2)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor('#cccccc')
        cell.set_linewidth(0.5)
        if r == 0:
            cell.set_text_props(weight='bold')
            cell.set_facecolor('#f2f2f2')

    fig.tight_layout(pad=0.5)

    out_path = f"df_standard_patches_table_sig{significant_digits}.png"
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved table image to: {out_path}")